<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/Taller_Control_1.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab">
</a>

# Taller domiciliario de control 1 — S1 a S6
**De datos crudos a una decisión explicable: SECOP + prensa + Cassandra + Neo4j**

**Individual · 6–8 horas · 100 puntos · Git/GitHub opcional · sin selección múltiple**

> 📘 **Antes de comenzar, lee la guía completa del taller:**  
> https://github.com/jazaineam1/BigData2026/blob/main/Talleres/Taller_Control_1.md

Este notebook es tu **espacio de trabajo y entrega**. La guía explica el caso, el propósito de cada etapa, el paso a paso, los productos esperados y la rúbrica.

### Reglas de trabajo

- No reemplaces código por capturas.
- No escribas manualmente resultados que pueden obtenerse a partir de los datos.
- Conserva las salidas de ejecución.
- Puedes consultar los cuadernos anteriores del curso.
- Las cuentas de Atlas, Astra y Aura son **opcionales** para este taller.
- Git/GitHub también es opcional.
- La evaluación se basa en los artefactos que produces y en controles reproducibles.


In [ ]:
from pathlib import Path
from datetime import date
import json, re, hashlib, urllib.request, subprocess, sys
import pandas as pd
import numpy as np

NOMBRE = input("Nombre completo: ").strip()
CODIGO = input("Código: ").strip()
if not NOMBRE or not CODIGO:
    raise ValueError("Completa nombre y código antes de continuar.")

OUT = Path("entrega_tc1")
OUT.mkdir(exist_ok=True)

# Los datos se fijan a una versión concreta para que el taller siga siendo reproducible
# aunque el repositorio principal cambie después.
DATA_COMMIT = "c7031e3a58daa22d4ceff2d2f01d66aa967ba9e3"
RAW = f"https://raw.githubusercontent.com/jazaineam1/BigData2026/{DATA_COMMIT}"

URLS = {
    "secop": f"{RAW}/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
    "noticias": f"{RAW}/Datos/noticias_contratacion_2026.json",
    "menciones": f"{RAW}/Datos/entidades_en_noticias_2026.json",
    "relacional": f"{RAW}/Datos/s06_contexto_relacional.csv",
    "manifest": f"{RAW}/Datos/s06_contexto_relacional_manifest.json",
}

print("Entorno listo.")
print("Datos fijados en el commit:", DATA_COMMIT[:12])
print("Carpeta de entrega:", OUT.resolve())


---
# ETAPA 1 — Ingesta, perfil de fuentes y arquitectura
**15 puntos**

Sigue la guía, sección **ETAPA 1**.

Debes dejar creados al terminar:

- `secop`
- `noticias`
- `menciones`
- `relacional`
- `manifest_s06`
- `perfil_fuentes`
- `arquitectura_inicial`

Y debes generar:

- `entrega_tc1/01_perfil_fuentes.json`

### Qué debe verse en el notebook

Para cada fuente muestra el tipo de objeto, tamaño, campos principales y un registro de ejemplo. Después construye el perfil y el diagrama Mermaid solicitado.


In [ ]:
# ============================================================
# TU TRABAJO — ETAPA 1
# ============================================================

secop = None
noticias = None
menciones = None
relacional = None
manifest_s06 = None

perfil_fuentes = {
    "secop": {},
    "noticias": {},
    "menciones": {},
    "relacional": {},
}

arquitectura_inicial = """
flowchart LR
    %% Completa aquí la arquitectura inicial solicitada en la guía.
"""

# 1. Carga las cinco fuentes usando URLS.
# 2. Muestra evidencia de la carga.
# 3. Calcula el perfil solicitado SIN escribir los números manualmente.
# 4. Guarda perfil_fuentes como OUT / "01_perfil_fuentes.json".
# 5. Completa arquitectura_inicial.


---
# ETAPA 2 — Construir una regla de priorización reproducible
**20 puntos**

Sigue la guía, sección **ETAPA 2**.

Debes dejar creados:

- `contexto_menciones`
- `paso1`
- `paso2`
- `paso3`
- `candidatos`
- `trazabilidad_regla`
- `bandeja`

Y generar:

- `entrega_tc1/02_bandeja_candidatos.csv`

### Idea central

No debes saltar directamente al resultado final. El evaluador debe poder reconstruir **cómo** se redujo el universo y comprobar que cada filtro corresponde a la regla definida.


In [ ]:
# ============================================================
# TU TRABAJO — ETAPA 2
# ============================================================

contexto_menciones = None
paso1 = None
paso2 = None
paso3 = None
candidatos = None
trazabilidad_regla = {}
bandeja = None

# Sigue exactamente los pasos de la guía:
# contexto_menciones -> paso1 -> paso2 -> paso3 -> candidatos -> bandeja
# Usa validate="many_to_one" en el merge.
# Guarda la bandeja como OUT / "02_bandeja_candidatos.csv".


---
# ETAPA 3 — Consultar evidencia documental con MongoDB
**20 puntos**

Sigue la guía, sección **ETAPA 3**.

Para que la nota no dependa de una cuenta externa se usa `mongomock`. Si quieres, después puedes repetir las consultas en Atlas como práctica adicional.

Debes dejar creados:

- `coleccion`
- `resultado_largas`
- `top_bogota`
- `pipeline_secciones`
- `resumen_secciones`
- `mongo_resultados`

Y generar:

- `entrega_tc1/03_mongo_resultados.json`


In [ ]:
# ============================================================
# PREPARACIÓN DEL MOTOR DOCUMENTAL LOCAL
# ============================================================

import importlib.util
if importlib.util.find_spec("mongomock") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mongomock"])

import mongomock

# ============================================================
# TU TRABAJO — ETAPA 3
# ============================================================

coleccion = None
resultado_largas = []
top_bogota = []
pipeline_secciones = []
resumen_secciones = []
mongo_resultados = {}

# 1. Crea una colección e inserta los documentos.
# 2. Implementa el filtro de noticias largas con proyección y orden.
# 3. Implementa top_bogota.
# 4. Implementa el pipeline de agregación.
# 5. Guarda mongo_resultados como OUT / "03_mongo_resultados.json".


---
# ETAPA 4 — Diseñar Cassandra desde la consulta
**15 puntos**

Sigue la guía, sección **ETAPA 4**.

La pregunta operacional es:

> Dado un `corte` y un `departamento`, devolver los 5 candidatos con mayor `valor_base`; en caso de empate, ordenar por `id_proceso` ascendente.

Debes dejar creados:

- `bandeja_operacional`
- `cql_create`
- `consulta_operacional`
- `departamento_prueba`
- `top5_operacional`

Y generar:

- `entrega_tc1/04_modelo_cassandra.cql`

La evaluación no pide memorizar CQL: pide demostrar que la clave primaria y el orden de clustering **sirven a la consulta solicitada**.


In [ ]:
# ============================================================
# TU TRABAJO — ETAPA 4
# ============================================================

bandeja_operacional = None

cql_create = """
-- Escribe aquí el CREATE TABLE solicitado en la guía.
"""

def consulta_operacional(df, corte, departamento, n=5):
    """
    Reproduce en pandas el patrón de acceso para el que diseñaste Cassandra.
    Debe devolver exactamente las filas de la partición solicitada y respetar
    el mismo orden de clustering.
    """
    return None

departamento_prueba = None
top5_operacional = None

# 1. Construye bandeja_operacional.
# 2. Escribe cql_create.
# 3. Implementa consulta_operacional.
# 4. Identifica por código el departamento con más candidatos.
# 5. Ejecuta el top 5.
# 6. Guarda cql_create como OUT / "04_modelo_cassandra.cql".


---
# ETAPA 5 — Agregar contexto relacional con Neo4j
**20 puntos**

Sigue la guía, sección **ETAPA 5**.

Debes dejar creados:

- `ancla`, `nit_ancla`
- `hist`, `hist_ancla`
- `prov_ancla`, `prov_global`
- `resultado_h2r`
- `maximo_h2r`
- `mediana_referencia`
- `cypher_contexto`
- `cypher_compartido`
- `cypher_ranking`
- `G`, `nodos_grafo`, `aristas_grafo`

Y generar:

- `entrega_tc1/05_consultas_neo4j.cypher`
- `entrega_tc1/06_contexto_relacional.csv`

Aquí no se evalúa que “el grafo se vea bonito”. Se evalúa que los **nodos, relaciones, consultas y métricas correspondan al problema relacional**.


In [ ]:
# ============================================================
# PREPARACIÓN DEL GRAFO LOCAL
# ============================================================

if importlib.util.find_spec("networkx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "networkx"])
import networkx as nx

# ============================================================
# TU TRABAJO — ETAPA 5
# ============================================================

ancla = None
nit_ancla = None
hist = None
hist_ancla = None
prov_ancla = None
prov_global = None
resultado_h2r = None
maximo_h2r = None
mediana_referencia = None

cypher_contexto = """
// Consulta 1 — contexto Entidad -> Proceso -> Proveedor
"""

cypher_compartido = """
// Consulta 2 — proveedor compartido con otras entidades
"""

cypher_ranking = """
// Consulta 3 — ranking por número de entidades DISTINCT conectadas
"""

G = nx.DiGraph()
nodos_grafo = 0
aristas_grafo = 0

# 1. Recupera el ancla desde manifest_s06, NO escribas el NIT manualmente.
# 2. Construye hist e hist_ancla.
# 3. Calcula prov_ancla, prov_global y resultado_h2r.
# 4. Calcula maximo_h2r y mediana_referencia.
# 5. Guarda resultado_h2r como OUT / "06_contexto_relacional.csv".
# 6. Escribe las tres consultas Cypher y guárdalas en "05_consultas_neo4j.cypher".
# 7. Construye el subgrafo NetworkX del ancla y actualiza nodos_grafo/aristas_grafo.


---
# ETAPA 6 — Integración, informe y paquete final
**10 puntos**

Sigue la guía, sección **ETAPA 6**.

Debes dejar creados:

- `arquitectura_final`
- `informe_tecnico`

Y generar:

- `entrega_tc1/07_informe_tecnico.md`

El informe no es una reflexión libre. Debe responder las siete secciones de la guía, usar los resultados calculados en las etapas anteriores y declarar explícitamente los límites de la evidencia.


In [ ]:
# ============================================================
# TU TRABAJO — ETAPA 6
# ============================================================

arquitectura_final = """
flowchart LR
    %% Completa la arquitectura final de punta a punta.
"""

informe_tecnico = f"""
## 1. Problema y decisión
[Completa]

## 2. Fuentes y calidad
[Completa]

## 3. Regla de priorización
[Completa]

## 4. Por qué MongoDB
[Completa]

## 5. Por qué Cassandra
[Completa]

## 6. Por qué Neo4j
[Completa]

## 7. Límites de la evidencia
[Completa]
"""

# Sustituye los marcadores por un informe técnico conciso y verificable.
# Debes incorporar mediante variables los números solicitados en la guía.
# Guarda el informe como OUT / "07_informe_tecnico.md".


---
# VALIDACIÓN Y ENTREGA

Ejecuta esta celda **solo cuando hayas terminado las seis etapas**.

El validador no evalúa estilo de redacción ni “opiniones correctas”. Comprueba objetos, conteos, consultas, estructura del modelo y artefactos. Al final genera:

- `entrega_tc1/manifest_tc1.json`
- puntaje / 100
- nota / 5.0
- SHA-256
- `TC1_<codigo>.zip`

Si un control falla, corrige la etapa correspondiente y vuelve a ejecutar el validador.


In [ ]:
# ============================================================
# VALIDADOR VERSIONADO — NO EDITAR
# ============================================================

VALIDATOR_COMMIT = "abffe079c8a64814281b2918b6a969e9bc7885e5"
VALIDATOR_URL = (
    "https://raw.githubusercontent.com/jazaineam1/BigData2026/"
    f"{VALIDATOR_COMMIT}/utils/tc1_validator.py"
)

codigo_validador = urllib.request.urlopen(VALIDATOR_URL).read().decode("utf-8")
espacio_validador = {}
exec(codigo_validador, espacio_validador)

manifest_tc1 = espacio_validador["evaluar"](globals())


## Checklist final del estudiante

Antes de entregar comprueba:

- [ ] ejecuté el notebook de arriba abajo;
- [ ] las seis etapas muestran sus resultados;
- [ ] `entrega_tc1/` contiene los siete productos técnicos;
- [ ] ejecuté el validador final;
- [ ] existe `manifest_tc1.json`;
- [ ] descargué `TC1_<codigo>.zip`;
- [ ] descargué también mi `.ipynb` con las salidas visibles;
- [ ] no usé una captura como sustituto del código o de un artefacto;
- [ ] no afirmé irregularidad, fraude, favorecimiento o colusión a partir de una señal que no lo demuestra.

**Git/GitHub sigue siendo opcional.** Si decides usarlo, entrega además la URL de tu commit como trazabilidad adicional, pero no cambia la nota del taller.
